In [1]:
import pandas as pd
import duckdb

In [2]:
dx_url = (
    "https://tuva-public-resources.s3.amazonaws.com/"
    "versioned_value_sets/latest/"
    "dxccsr_v2023_1_cleaned_map_compressed.csv.gz"
)
pr_url = (
    "https://tuva-public-resources.s3.amazonaws.com/"
    "versioned_value_sets/latest/"
    "prccsr_v2023_1_cleaned_map.csv_0_0_0.csv.gz"
)
cms_url = (
    "https://tuva-public-resources.s3.amazonaws.com/"
    "versioned_value_sets/latest/"
    "cms_chronic_conditions_hierarchy.csv_0_0_0.csv.gz"
)

In [3]:
expected_dx = [
    'icd_10_cm_code',
    'icd_10_cm_code_description',
    'default_ccsr_category_ip',
    'default_ccsr_category_description_ip',
    'default_ccsr_category_op',
    'default_ccsr_category_description_op',
    'ccsr_category_1',
    'ccsr_category_1_description',
    'ccsr_category_2',
    'ccsr_category_2_description',
    'ccsr_category_3',
    'ccsr_category_3_description',
    'ccsr_category_4',
    'ccsr_category_4_description',
    'ccsr_category_5',
    'ccsr_category_5_description',
    'ccsr_category_6',
    'ccsr_category_6_description'
]

expected_pr = [
    'icd_10_pcs',
    'icd_10_pcs_description',
    'prccsr',
    'prccsr_description',
    'clinical_domain'
]

expected_cms = [
    'condition_id',
    'condition',
    'condition_column_name',
    'chronic_condition_type',
    'condition_category',
    'additional_logic',
    'claims_qualification',
    'inclusion_type',
    'code_system',
    'code'
]

In [4]:
df_dx = pd.read_csv(
    dx_url,
    compression='gzip',
    header=None,
    names=expected_dx,
    dtype=str,
    usecols=range(len(expected_dx))
)

df_dx.head()

,icd_10_cm_code,icd_10_cm_code_description,default_ccsr_category_ip,default_ccsr_category_description_ip,default_ccsr_category_op,default_ccsr_category_description_op,ccsr_category_1,ccsr_category_1_description,ccsr_category_2,ccsr_category_2_description,ccsr_category_3,ccsr_category_3_description,ccsr_category_4,ccsr_category_4_description,ccsr_category_5,ccsr_category_5_description,ccsr_category_6,ccsr_category_6_description
0,A000,"Cholera due to Vibrio cholerae 01, biovar chol...",DIG001,Intestinal infection,DIG001.1,Intestinal infection.1,DIG001.2,Intestinal infection.2,INF003,Bacterial infections,\N,\N.1,\N.2,\N.3,\N.4,\N.5,\N.6,\N.7
1,A001,"Cholera due to Vibrio cholerae 01, biovar eltor",DIG001,Intestinal infection,DIG001,Intestinal infection,DIG001,Intestinal infection,INF003,Bacterial infections,\N,\N,\N,\N,\N,\N,\N,\N
2,A009,"Cholera, unspecified",DIG001,Intestinal infection,DIG001,Intestinal infection,DIG001,Intestinal infection,INF003,Bacterial infections,\N,\N,\N,\N,\N,\N,\N,\N
3,A0100,"Typhoid fever, unspecified",DIG001,Intestinal infection,DIG001,Intestinal infection,DIG001,Intestinal infection,INF003,Bacterial infections,\N,\N,\N,\N,\N,\N,\N,\N
4,A0101,Typhoid meningitis,NVS001,Meningitis,NVS001,Meningitis,INF003,Bacterial infections,NVS001,Meningitis,\N,\N,\N,\N,\N,\N,\N,\N


In [5]:
df_pr = pd.read_csv(
    pr_url,
    compression='gzip',
    header=None,
    names=expected_pr,
    dtype=str,
    usecols=range(len(expected_pr))
)

df_pr.head()

,icd_10_pcs,icd_10_pcs_description,prccsr,prccsr_description,clinical_domain
0,0016070,Bypass Cerebral Ventricle to Nasopharynx with ...,CNS010,Cerebrospinal fluid shunt procedures,Central Nervous System Procedures
1,0016071,Bypass Cerebral Ventricle to Mastoid Sinus wit...,CNS010,Cerebrospinal fluid shunt procedures,Central Nervous System Procedures
2,0016072,Bypass Cerebral Ventricle to Atrium with Autol...,CNS010,Cerebrospinal fluid shunt procedures,Central Nervous System Procedures
3,0016073,Bypass Cerebral Ventricle to Blood Vessel with...,CNS010,Cerebrospinal fluid shunt procedures,Central Nervous System Procedures
4,0016074,Bypass Cerebral Ventricle to Pleural Cavity wi...,CNS010,Cerebrospinal fluid shunt procedures,Central Nervous System Procedures


In [6]:
df_cms = pd.read_csv(
    cms_url,
    compression='gzip',
    header=None,
    names=expected_cms,
    dtype=str,
    usecols=range(len(expected_cms))
)

df_cms.head()

,condition_id,condition,condition_column_name,chronic_condition_type,condition_category,additional_logic,claims_qualification,inclusion_type,code_system,code
0,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I230
1,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I231
2,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I232
3,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I233
4,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I234


In [7]:
conn = duckdb.connect(database='/workspaces/txwc/tx_workers_comp.db', read_only=False)

In [8]:
conn.execute("CREATE SCHEMA IF NOT EXISTS ccsr;")

In [9]:
conn.register("temp_dx", df_dx)
conn.execute("""
CREATE OR REPLACE TABLE ccsr._value_set_dxccsr_v2023_1_cleaned_map AS
SELECT * FROM temp_dx;
""")

In [10]:
conn.register("temp_pr", df_pr)
conn.execute("""
CREATE OR REPLACE TABLE ccsr._value_set_prccsr_v2023_1_cleaned_map AS
SELECT * FROM temp_pr;
""")

In [11]:
conn.register("temp_cms", df_cms)
conn.execute("""
CREATE OR REPLACE TABLE chronic_conditions._value_set_cms_chronic_conditions_hierarchy AS
SELECT * FROM temp_cms;
""")

In [12]:
conn.close()